# k-NN, Naive Bayes and SVM — Solution

**Dataset:** Titanic passengers; numerical inputs for k-NN/SVM and sex/class for Naive Bayes.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

## 1. Compute one neighbour distance

Replace the gap with `np.sqrt(np.sum((new_point - known_point) ** 2))`. If 3 of 5 neighbours are positive, what is the class probability?

In [2]:
new_point = np.array([2, 3])
known_point = np.array([5, 7])
distance = np.sqrt(np.sum((new_point - known_point) ** 2))
print('Distance:', distance)
print('Positive probability:', 3 / 5)

Distance: 5.0
Positive probability: 0.6


**Answer:** The distance is 5 and the positive-class probability is 0.6.

- Differences: 5 − 2 = 3 and 7 − 3 = 4.
- Distance: √(3² + 4²) = √(9 + 16) = √25 = 5.
- With 3 positive neighbours out of 5, the probability is 3 / 5 = 0.6.
- 0.6 is above 0.5, so the majority class is positive.

## 2. Prepare the data

Preparation is provided. Which inputs have different units? Why must the target be excluded from input distances?

In [3]:
titanic = pd.read_csv('../00_Data/titanic.csv')
# Fixed, simple preparation: use rows with known age and fare.
# Removing missing rows may change which passengers are represented.
titanic = titanic.dropna(subset=['Age', 'Fare']).copy()
X = titanic[['Age', 'Fare', 'Pclass']].copy()
X['male'] = (titanic['Sex'] == 'male').astype(int)
y = titanic['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.head())
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

      Age     Fare  Pclass  male
619  26.0  10.5000       2     1
622  20.0  15.7417       3     1
72   21.0  73.5000       2     1
339  45.0  35.5000       1     1
513  54.0  59.4000       1     0
Training rows: 571 Test rows: 143


**Answer:** Age (years), fare (pounds) and class (1–3) have different units and ranges; `male` is 0/1.

- Fares range up to several hundred pounds, while class is only 1, 2 or 3 and `male` only 0 or 1. Without scaling, fare would dominate every distance.
- The target `Survived` must not be an input. A new passenger's survival is unknown, and using it would reveal the answer to the model.
- As in module 05, 571 passengers are used for training and 143 for testing.

## 3. Train a scaled k-NN

Replace the gap with `KNeighborsClassifier(n_neighbors=5)`. The scaler is provided inside a pipeline. Run CV before fitting and final test evaluation.

In [4]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
print('Scaled k-NN CV accuracy:', round(scores.mean(), 3))
knn.fit(X_train, y_train)
print('Fixed k=5 test accuracy:', round(knn.score(X_test, y_test), 3))
new_passenger = pd.DataFrame({'Age': [30], 'Fare': [80], 'Pclass': [1], 'male': [0]})
print('P(survived):', knn.predict_proba(new_passenger)[:, 1])

Scaled k-NN CV accuracy: 0.807
Fixed k=5 test accuracy: 0.776
P(survived): [1.]


**Answer:**

- **CV accuracy (k = 5):** 0.807. The pipeline learns the scaling inside each CV fold, so the validation fold never influences the scaling.
- **Test accuracy:** 0.776, so 111 of 143 test passengers are classified correctly. Predicting "not survived" for everyone would give 85 / 143 ≈ 0.594.
- **New passenger** (woman, 30 years, 1st class, fare 80): P(survived) = 1.0, so all 5 nearest neighbours survived.
- The fitted model scales the new passenger with the training means and standard deviations.
- A probability of 1.0 based on only five neighbours is a strong local majority, not certainty.

## 4. See scaling and the choice of k

The comparison is supplied. Compare scaled and unscaled CV accuracy, then inspect only three k values. Do not use test scores to choose k.

In [5]:
unscaled = KNeighborsClassifier(n_neighbors=5)
print('Unscaled CV accuracy:', round(cross_val_score(
    unscaled, X_train, y_train, cv=5, scoring='accuracy').mean(), 3))
for k in [3, 5, 15]:
    candidate = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    score = cross_val_score(candidate, X_train, y_train, cv=5, scoring='accuracy').mean()
    print('k:', k, 'scaled CV accuracy:', round(score, 3))

Unscaled CV accuracy: 0.678
k: 3 scaled CV accuracy: 0.813
k: 5 scaled CV accuracy: 0.807


k: 15 scaled CV accuracy: 0.786


**Answer:** Scaling helps here: with k = 5, the scaled CV accuracy is 0.807 compared with 0.678 unscaled.

- That is 13 percentage points better with scaling. Without scaling, distances are dominated by fare, so the neighbours are mainly passengers with similar fares.
- Scaling changes distances, so it changes which neighbours are found. It does not raise accuracy on every dataset.
- Among the three scaled candidates, k = 3 has the highest CV accuracy (0.813), ahead of k = 5 (0.807) and k = 15 (0.786).
- The differences between k = 3 and k = 5 are small (0.006). With k = 15, the predictions become too smooth.
- CV can guide k; the fixed k = 5 example above is not changed afterwards based on its test score.

## 5. Work a Naive Bayes example

Use the supplied scores from the golf example. Replace the gap with `no / (yes + no)`. Which class has the larger score?

In [6]:
yes = (9 / 14) * (2 / 9) * (3 / 9) * (3 / 9) * (3 / 9)
no = (5 / 14) * (3 / 5) * (1 / 5) * (4 / 5) * (3 / 5)
p_no = no / (yes + no)
print('Scores yes/no:', round(yes, 4), round(no, 4))
print('P(no):', round(p_no, 3))

Scores yes/no: 0.0053 0.0206
P(no): 0.795


**Answer:** "No" has the larger score, and P(no) ≈ 0.795.

- score(yes) = (9/14)(2/9)(3/9)(3/9)(3/9) ≈ 0.0053.
- score(no) = (5/14)(3/5)(1/5)(4/5)(3/5) ≈ 0.0206.
- Sum of both scores: 0.0053 + 0.0206 = 0.0259.
- P(no) = 0.0206 / 0.0259 ≈ 0.795; P(yes) ≈ 0.205.
- Multiplying the single probabilities is only allowed under the conditional-independence assumption: given the class, the weather inputs are treated as independent.

## 6. Apply categorical Naive Bayes

The code is provided. Here passenger class and sex are categories, not continuous distances. Why does this model use fewer inputs than k-NN?

In [7]:
cat_train = X_train[['male', 'Pclass']].copy()
cat_test = X_test[['male', 'Pclass']].copy()
cat_train['Pclass'] = cat_train['Pclass'] - 1
cat_test['Pclass'] = cat_test['Pclass'] - 1
nb = CategoricalNB(min_categories=[2, 3])
nb.fit(cat_train, y_train)
print('Naive Bayes test accuracy:', round(nb.score(cat_test, y_test), 3))
woman_first_class = pd.DataFrame({'male': [0], 'Pclass': [0]})
print('P(survived):', nb.predict_proba(woman_first_class)[:, 1].round(3))

Naive Bayes test accuracy: 0.783
P(survived): [0.891]


**Answer:** `CategoricalNB` counts how often each category appears in each class, so it needs categorical inputs.

- Sex and class are clearly categorical, so this example uses only these two inputs.
- Age and fare are continuous and would need another modelling choice, such as binning into age groups.
- **Test accuracy:** 0.783 with only sex and class. This is similar to k-NN's 0.776 with four inputs, so most of the predictive information lies in sex and class.
- **Woman in 1st class:** P(survived) = 0.891. The depth-2 tree in module 05 gave 0.946 and k-NN gave 1.0 for a similar passenger.
- The test scores are not a fair comparison of methods, because the models use different inputs.

## 7. Train a simple SVM

Replace the gap with `SVC(kernel="linear", C=1)`. The fixed model is evaluated once; the two extra kernel examples use training CV only. Which kernel allows a curved boundary?

In [8]:
svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1))
svm.fit(X_train, y_train)
print('Fixed linear SVM test accuracy:', round(svm.score(X_test, y_test), 3))
print('Number of support vectors:', svm[-1].n_support_.sum())
for kernel in ['linear', 'rbf']:
    candidate = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1))
    scores = cross_val_score(candidate, X_train, y_train, cv=5, scoring='accuracy')
    print(kernel, 'CV accuracy:', round(scores.mean(), 3))

Fixed linear SVM test accuracy: 0.783
Number of support vectors: 275


linear CV accuracy: 0.779


rbf CV accuracy: 0.797


**Answer:** The RBF kernel allows a curved boundary; the linear kernel uses a straight boundary (a flat plane) in the scaled inputs.

- **Fixed linear SVM:** test accuracy 0.783.
- **Support vectors:** 275 of 571 training passengers (48%). These are the rows on or inside the margin, and they alone determine the boundary. Such a high share shows that survivors and non-survivors overlap strongly.
- **CV accuracy:** linear 0.779, RBF 0.797. The curved RBF boundary fits these validation folds about 2 percentage points better.
- Kernel choices should be compared using CV, as here; no single kernel is always best.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [9]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))

Linux-6.8.0-1064-azure-x86_64-with-glibc2.41
Python: 3.11.16
Date: 2026-10-02T20:17:25
